In [4]:
import os
import re
import sys
import json
import torch
import spacy
import pickle
import contextlib
import numpy as np
from tqdm import tqdm
from typing import List, Tuple, Dict

from qiskit_aer import AerSimulator
from pytket.extensions.qiskit.backends.aer import AerBackend
# from qiskit.providers.aer import AerSimulator
# from pytket.extensions.qiskit import AerBackend


from lambeq.backend.grammar import Diagram, Id
from lambeq import (
    AtomicType,
    IQPAnsatz,
    RemoveCupsRewriter,
    SimpleRewriteRule,
    Rewriter,
    UnifyCodomainRewriter,
    DepCCGParser
)

from collections.abc import Mapping, Sequence
import statistics

# import depccg
# from lambeq import ( CCGParser, CCGTree, CCGRuleUseError, CCGRule, CCGType,
#                     CCGBankParseError, CCGBankParser, DepCCGParseError )


In [5]:
import logging
logging.getLogger("allennlp").setLevel(logging.WARNING)
logging.getLogger("depccg").setLevel(logging.WARNING)
parser = DepCCGParser(model='elmo', device=0) # device=:  -1 == CPU | 0 == GPU | 1 == second GPU

2026-05-03 12:51:31,789 - INFO - cached_path - cache of https://s3-us-west-2.amazonaws.com/allennlp/models/elmo/2x4096_512_2048cnn_2xhighway/elmo_2x4096_512_2048cnn_2xhighway_options.json is up-to-date
2026-05-03 12:51:32,443 - INFO - cached_path - cache of https://s3-us-west-2.amazonaws.com/allennlp/models/elmo/2x4096_512_2048cnn_2xhighway/elmo_2x4096_512_2048cnn_2xhighway_weights.hdf5 is up-to-date
2026-05-03 12:51:33,157 - INFO - cached_path - cache of https://s3-us-west-2.amazonaws.com/allennlp/models/elmo/2x4096_512_2048cnn_2xhighway/elmo_2x4096_512_2048cnn_2xhighway_weights.hdf5 is up-to-date
2026-05-03 12:51:33,862 - INFO - cached_path - cache of https://s3-us-west-2.amazonaws.com/allennlp/models/elmo/2x4096_512_2048cnn_2xhighway/elmo_2x4096_512_2048cnn_2xhighway_weights.hdf5 is up-to-date
2026-05-03 12:51:34,568 - INFO - cached_path - cache of https://s3-us-west-2.amazonaws.com/allennlp/models/elmo/2x4096_512_2048cnn_2xhighway/elmo_2x4096_512_2048cnn_2xhighway_weights.hdf5 is u

In [6]:
def get_deep_type(obj):
    if isinstance(obj, list):
        # We look at the unique types inside the list to keep it readable
        inner_types = {get_deep_type(item) for item in obj}
        return f"List[{' | '.join(sorted(inner_types))}]"

    elif isinstance(obj, dict):
        # We summarize the types of all keys and all values
        key_types = {get_deep_type(k) for k in obj.keys()}
        val_types = {get_deep_type(v) for v in obj.values()}
        return f"Dict[{' | '.join(sorted(key_types))}, {' | '.join(sorted(val_types))}]"

    else:
        # Return the class name (e.g., 'Diagram' or 'str')
        return type(obj).__name__

def get_deep_shape(obj, level=0):
    indent = "  " * level
    
    # 1. Atomic types (Strings/Bytes) - Check these first 
    # because they are technically Sequences too!
    if isinstance(obj, (str, bytes)):
        return f"{indent}str"

    # 2. Dictionaries (Mappings)
    elif isinstance(obj, Mapping):
        if not obj:
            return f"{indent}dict(len=0)"

        lines = [f"{indent}dict(len={len(obj)})"]
        for key, value in obj.items():
            # Get child shape and strip only the first line's indentation 
            # so we can prefix it with our key label
            child = get_deep_shape(value, level + 1).lstrip()
            lines.append(f"{indent}  key={repr(key)} -> {child}")
        return "\n".join(lines)

    # 3. Sequences (Lists, Tuples, etc.)
    elif isinstance(obj, Sequence):
        name = type(obj).__name__
        if not obj:
            return f"{indent}{name}(len=0)"

        header = f"{indent}{name}(len={len(obj)})"
        
        # Calculate shapes of all children
        child_shapes = [get_deep_shape(item, level + 1).lstrip() for item in obj]
        unique_shapes = sorted(list(set(child_shapes)))

        if len(unique_shapes) == 1:
            # All items are identical structure
            return f"{header}\n{indent}  [*] -> {unique_shapes[0]}"
        else:
            lines = [header]
            for i, shape in enumerate(child_shapes):
                lines.append(f"{indent}  [{i}] -> {shape}")
            return "\n".join(lines)

    # 4. Base objects (int, float, None, etc.)
    else:
        return f"{indent}{type(obj).__name__}"

def find_mismatches(data_a: List, data_b: List):
    # 1. Check if the outer lists are even the same length
    if len(data_a) != len(data_b):
        print(
            f"❌ [CRITICAL] Outer List Length Mismatch: List A={len(data_a)}, List B={len(data_b)}"
        )

    # Iterate through the top-level list
    for i, (dict_a, dict_b) in enumerate(zip(data_a, data_b)):
        # Check if the keys in the dictionaries match
        keys_a = set(dict_a.keys())
        keys_b = set(dict_b.keys())

        if keys_a != keys_b:
            print(f"❌ [Index {i}] Key Mismatch:")
            print(f"   Keys only in A: {keys_a - keys_b}")
            print(f"   Keys only in B: {keys_b - keys_a}")
            continue  # Skip to next list item if keys don't match

        # Check values for each key
        for key in keys_a:
            list_a = dict_a[key]
            list_b = dict_b[key]

            # 2. Check lengths of the lists inside the dictionary
            if len(list_a) != len(list_b):
                print(f"❌ [Index {i}][Key: '{key}'] Inner List Length Mismatch:")
                print(f"   Length A: {len(list_a)}")
                print(f"   Length B: {len(list_b)}")

            # 3. Check individual elements inside those lists
            # This handles List[Diagram], List[List[int]], and List[str]
            for j, (val_a, val_b) in enumerate(zip(list_a, list_b)):
                if val_a != val_b:
                    print(
                        f"❌ [Index {i}][Key: '{key}'][Inner Index {j}] Content Mismatch!"
                    )
                    print(f"   Type A: {type(val_a).__name__}")
                    print(f"   Type B: {type(val_b).__name__}")

                    # If they are small (like List[int] or str), print the actual value
                    if not hasattr(
                        val_a, "draw"
                    ):  # Don't print full Diagrams, they are too big
                        print(f"   Value A: {val_a}")
                        print(f"   Value B: {val_b}")
                    else:
                        print(
                            f"   (Diagram content differs - possibly different boxes or wires)"
                        )

def diagnose_variable(variable):
    # print("TOP LEVEL LENGTH:", len(variable) if hasattr(variable, '__len__') else "N/A")
    print("DEEP TYPE:")
    print(get_deep_type(variable))
    print("\nDEEP SHAPE:")
    print(get_deep_shape(variable))



def print_length(dataset: List[Dict]):
    dict_string = "text_sentences"
    if "circuits" in list(dataset[0].keys()):
        dict_string = "circuits"

    count = 0
    count_labels = 0

    for data in dataset:
        count += len(data[dict_string])
        count_labels += len(data["labels"])
    
    print(f"n_{dict_string}: {count} | n_labels: {count_labels}")

def print_errors(errors: List):
    print(f"error_list_length: {len(errors)}")
    for i, error in enumerate(errors):
        print(f"{i}: {error}")

def print_n_qubits(n_qubits_list: List[int]):
    print(f"qubit_list: {n_qubits_list}")
    print(f"qubit_list_length: {len(n_qubits_list)}")
    print(f"qubit_list_mean: {statistics.mean(n_qubits_list)}\n")
    


In [30]:
def combine_articles(dataset: List[Dict]) -> List[Dict]:
    combined_text_sentences, combined_labels = [], []

    for article in dataset:
        # zip(combined, article["text_sentences"]) ####### paklaust gpt kam naudojamas zip ir kodel jis cia neveikia, i.e. kaip jis cia tiksliai veikia?
        # zip(labels, article["labels"])

        combined_text_sentences += article["text_sentences"]
        combined_labels += article["labels"]

    combined_dataset = [{
        "text_sentences": combined_text_sentences,
        "labels": combined_labels,
    }]

    return combined_dataset

def combine_n_articles(dataset: List[Dict], n_to_merge: int = 3) -> List[Dict]:
    combined_dataset = []
    
    for i in range(0, len(dataset), n_to_merge):
        chunk = dataset[i : i + n_to_merge]
        combined_dataset.extend(combine_articles(chunk))
        
    return combined_dataset

def load_PreSumm_pts(file_number: int=0, ds_purpose: str="train", calculate_articles: bool=False) -> List[Dict] | Tuple[List[Dict], List[int], List[int], List[int]]:
    valid_ds, elements_valid_labels = [], []
    n_sentences, n_labels, n_labels_true = [], [], []
    
    print(f"load_PreSumm_pts_{file_number}")
    file_path = f"Dataset/Raw/cnn_dailymail/_PreSumm/cnndm.{ds_purpose}.{file_number}.bert.pt"
    loaded_data = torch.load(file_path)
    
    elements_valid_labels = remove_negative_articles(loaded_data, "src_txt", "src_sent_labels", 1)

    for i, file_article in enumerate(elements_valid_labels):
        quantum_state_distribution_labels = []
        for label in file_article["src_sent_labels"]:
            if label:
                quantum_state_distribution_labels.append([0,1])
            else:
                quantum_state_distribution_labels.append([1,0])

        valid_ds.append(
            {
                "text_sentences": file_article["src_txt"],
                # "org_labels": line["src_sent_labels"],
                "labels": quantum_state_distribution_labels
            }
        )
        n_sentences.append(len(file_article["src_txt"]))
        n_labels.append(len(file_article["src_sent_labels"]))
        n_labels_true.append(file_article["src_sent_labels"].count(1))

    if calculate_articles:
        n_sentences_sum = sum(n_sentences)
        labes_sum       = sum(n_labels)
        labels_true_sum = sum(n_labels_true)
        print(f"\n n_sentences: {n_sentences_sum} \n n_labels: {labes_sum} (average {labes_sum/len(n_labels):.4f}) \n n_labels_true: {labels_true_sum} (average {labels_true_sum/len(n_labels_true):.4f})")
    
    return valid_ds

def read_PreSum_multiple(amount: int=4, dataset_purpose: str="train"):
    HIGHEST_FILENAME_NUMBER = 8
    left = 0
    right = amount
    loaded_combined_dataset = []

    while left < HIGHEST_FILENAME_NUMBER:
        print(f"read_PreSumm_multiple_{left}")
        loaded_combined_dataset.append(
            combine_articles( load_PreSumm_pts(left, right, dataset_purpose) )
        )
        left = right
        right = min(HIGHEST_FILENAME_NUMBER, left + amount)

    return loaded_combined_dataset


In [8]:
# https://chatgpt.com/s/t_69b2c7f7e28081919384bb842e7c46b3 - apie sakiniu preprocesinga kuris sutrumpina 40-60 proc sakiniu ilgio

ansatz    = IQPAnsatz(
    {AtomicType.SENTENCE: 1,
     AtomicType.NOUN:     1, # Galima priskirti 2 qubitus, jei, pvz, treniravimo rezultatai yra prasti
     }, # AtomicType.PREPOSITIONAL_PHRASE: 0,
    n_layers=2, n_single_qubit_params=3
)

def create_rewriter():
    # Rule to delete conjunction boxes (“and”, “but”) # just the wire, no box
    conj_rule = SimpleRewriteRule(cod=AtomicType.CONJUNCTION, template=Id(AtomicType.CONJUNCTION))

    # Rule to delete punctuation boxes (commas, quotes, dashes)
    # punc_rule = SimpleRewriteRule(cod=AtomicType.PUNCTUATION, template=Id(AtomicType.PUNCTUATION))

    # remove_pp2 = SimpleRewriteRule(cod=AtomicType.PREPOSITIONAL_PHRASE, template=Id(AtomicType.SENTENCE))
    # remove_pp = SimpleRewriteRule(cod=AtomicType.PREPOSITIONAL_PHRASE, template=Id(AtomicType.NOUN))

    # rewriter = Rewriter(
    #     [
    #         'coordination', 'determiner',
    #         'postadverb', 'preadverb',
    #         'connector', 'auxiliary',
    #         'prepositional_phrase',
    #         'subject_rel_pronoun',
    #         'object_rel_pronoun',
    #         'adjective', 'noun_phrase',
    #     ]
    # )

    rewriter = Rewriter([
        'determiner',
        'auxiliary',
        'connector',
        'coordination',
        'prepositional_phrase', 
        # 'subject_rel_pronoun', # They don't hurt performance, and they act as a safety
        # 'object_rel_pronoun'   # net in case the spaCy parser misses a relative clause.
    ])
    
    rewriter.add_rules(conj_rule)
    return rewriter

rewriter = create_rewriter()
remove_cups = RemoveCupsRewriter()
unify = UnifyCodomainRewriter(output_type=AtomicType.SENTENCE)

simulator = AerSimulator(
    method="statevector", device="GPU",
    precision="single",         # 32-bit float for ~2× speedup on large statevectors
    cuStateVec_enable=True  ,     # turn on NVIDIA cuStateVec kernels
    # batched_shots_gpu=True,     # batch thousands of shots very efficiently on GPU
    # batched_shots_gpu_max_qubits=29, # 8 ⋅ 2^n == 7 ⋅ 2^30, n ~= 29.8
    # num_threads_per_device=2    # limit CPU threads per GPU to reduce overhead
)

backend = AerBackend(simulation_method="statevector")
backend._qiskit_backend = simulator

# comp_pass = backend.default_compilation_pass(2)

/home/green/QNLPModelTraining/qnlp_3_10/lib/python3.10/site-packages/pytket/extensions/qiskit/backends/aer.py:129: UserWarning: More than one backend with name 'aer_simulator' is available. Picking one.
  warnings.warn(


In [20]:
CLEAN_REGEX = re.compile(r"[^\w\s'-]")

# CORE_DEPS = {"ROOT", "nsubj", "dobj", "pobj", "attr"}

# https://chatgpt.com/s/t_69c6b7d3545c8191a4935319498eee59  -> shows and explains how to use spacy to remove `appos`, `acl`, `advcl`, etc. relations
REMOVE_DEPS = {
    "appos",  # appositive
    "acl",    # clausal modifier
    "advcl",  # adverbial clause
    "relcl",  # relative clause
    # "amod",   # Adjectival Modifier (optional if the circuits are still to big)
    # "xcomp",   # open clausal complement | Probably do not use
    # "ccomp"    # clausal complement | Probably do not use

}

# https://chatgpt.com/s/t_69cd50d9e7848191aca0c6ccd42a1ff8 -> expains "ner", "textcat", "lemmatizer", "tagger", "parser"
nlp = spacy.load(
    # "en_core_web_sm",
    "en_core_web_trf",
    disable=["ner", "textcat", "lemmatizer", "tagger"],  # keep only parser
)

def sentence_simplify_spacy(sentences: List[str]) -> Tuple[List[str], List[int]]:
    results = []
    removed_idx = []

    for idx, doc in enumerate(nlp.pipe(sentences, batch_size=128)):
        to_remove = set()

        # Step 1: Identify subordinate clauses to remove
        for token in doc:
            if token.dep_ in REMOVE_DEPS:
                to_remove.update(t.i for t in token.subtree)

        # Step 2: Rebuild sentence
        pruned = "".join(
            token.text_with_ws
            for token in doc
            if token.i not in to_remove
        )

        # Step 3: Clear symbols
        cleaned = clean_sentence(pruned)
        # Step 4: Remove multi-spaces
        cleaned = " ".join(cleaned.split())

        if cleaned:
            results.append(cleaned)
        else:
            removed_idx.append(idx)

    print(f"In pruning lost {len(removed_idx)} out of {len(sentences)} sentences.")
    return results, removed_idx

def clean_sentence(sent: str) -> str:
    return CLEAN_REGEX.sub("", sent)
    # def clean_sentence(sentences):
    #     return [CLEAN_REGEX.sub("", s) for s in sentences if s is not None]

def remove_negative_articles(dataset: List[Dict], sentences_key: str, labels_key: str,
    positive_label, min_positive_ratio: float = 0.10):
    invalid_ratio, valid_articles = [], []
    one_positive_low_ratio, two_positive_low_ratio = [], []
    zero_positive = []
    label_distribution = {
        1: 0,
        2: 0,
        3: 0,
        4: 0,
        ">4": 0
    }

    for i, article in enumerate(dataset):
        sentences = article.get(sentences_key)
        labels = article.get(labels_key)

        if sentences is None or labels is None:
            invalid_ratio.append(i)
            continue

        n_labels = len(labels)

        if len(sentences) != len(labels):
            invalid_ratio.append(i)
            continue

        if n_labels == 0:
            invalid_ratio.append(i)
            continue

        positive_count = labels.count(positive_label)
        positive_ratio = positive_count / n_labels

        if positive_count == 0:
            zero_positive.append(i)
            continue

        elif positive_count in label_distribution:
            label_distribution[positive_count] += 1
        else:
            label_distribution[">4"] += 1
        

        if n_labels > 12 and positive_count == 1:
            one_positive_low_ratio.append(i)
        elif n_labels > 20 and positive_count == 1:
            two_positive_low_ratio.append(i)

        valid_articles.append(article)

    print(f"No+: {len(zero_positive)} | 1+ (>12): {len(one_positive_low_ratio)} | 2+ (>30): {len(two_positive_low_ratio)}")
    print(f"Label distribution: 1: {label_distribution[1]} | 2: {label_distribution[2]} | 3: {label_distribution[3]} | 4: {label_distribution[4]} | >4: {label_distribution['>4']}")
    print(f"Positive labels ratio: {positive_ratio:.4f}")
    print(f"Invalid ratio: {len(invalid_ratio)} out of {len(dataset)} articles.")

    return valid_articles

    

In [19]:
def remove_by_idx(ls: List, remove: List[int]):
    if not remove:
        return ls
    for n in reversed(remove):
        try:
            ls.pop(n)
        except Exception as e:
            print(f"Error removing index {n} from list of length {len(ls)}:   {e}")
    # print("remove value:", remove)
    return ls

def sent2diagrams_single(sentences: List[str]):
    diagrams, none_idx = [], []
    for i, sent in enumerate(sentences):
        d = parser.sentence2diagram(sent, tokenised=False, suppress_exceptions=True)
        if d is None:
            none_idx.append(i)
            continue

        diagrams.append(d)
    print(f"In sentence2diagrams_single() lost {len(sentences) - len(diagrams)} out of {len(sentences)}.")

    return diagrams, none_idx

def sent2diagrams(sentences: List[str]):
    none_idx = []
    
    diagrams = parser.sentences2diagrams(
        sentences, tokenised=False, suppress_exceptions=True
    )
    for i, diag in enumerate(diagrams):
        if diag is None:
            none_idx.append(i)

    print(f"In sentences2diagrams() lost {len(none_idx)} out of {len(sentences)}.")

    return diagrams, none_idx

def normalize(sentence_diagrams: List[Diagram]):
    diagrams_normalized, none_idx, errs = [], [], []
    drop_rewrite = 0
    drop_cups = 0
    for i, d in enumerate(sentence_diagrams):
        try:
            d = rewriter(d)
            if d is None:
                none_idx.append(i)
                drop_rewrite += 1
                continue

            d = remove_cups(d)
            d = d.normal_form()
            d = d.pregroup_normal_form()
            d = unify(d)

        except Exception as e:
            none_idx.append(i)
            errs.append(f"normalize() | {e}")
            drop_cups += 1
            continue

        diagrams_normalized.append(d)

    print(f"In normalize() lost {drop_rewrite} (rewrite) and {drop_cups} (cup removal) out of {len(sentence_diagrams)}.")

    return diagrams_normalized, none_idx, errs

def quantum_encode(diagrams: List[Diagram]):
    encoded_diagrams, remove, errs = [], [], []
    for i, diagram in enumerate(diagrams):
        try:
            circ = ansatz(diagram)
            encoded_diagrams.append(circ)
        except Exception as e:
            errs.append(f"quantum_encode() | {e}")
            remove.append(i)

    print(f"In quantum_encode() lost {len(remove)} out of {len(diagrams)}.")
    print(len(errs))

    return encoded_diagrams, remove, errs

def will_train(
    circuits: List,
    qubit_limit: int=28,
    # mem_limit_bytes: int = 7 * 2**30,
    mem_limit_bytes: int=16 * 2**28,
    circuit_depth_limit: int=200,
    gate_limit: int=20000,
):

    valid, invalid_idxs, errs = [], [], []
    n_qubits_list = []

    for idx, circ in enumerate(circuits):
        try:
            tk_circ = circ.to_tk()
            n_qubits = tk_circ.n_qubits
            n_qubits_list.append(n_qubits)

            if n_qubits > qubit_limit:
                needed = 16 * (2**n_qubits)
                raise RuntimeError(
                    f"Needs {needed} bytes > limit {mem_limit_bytes} bytes ({needed / 2**20:.0f} MiB > {(mem_limit_bytes / 2**20):.0f} MiB). "
                )
            
            # even when samll amount of qubits, there could be a lot of gates and circuit depth
            # which will slow down training process
            if tk_circ.depth() > circuit_depth_limit:
                raise RuntimeError(f"Circuit too deep: {tk_circ.depth()}")
            
            if tk_circ.n_gates > gate_limit:
                raise RuntimeError(f"Circuit contains too many gates: {tk_circ.n_gates}")

            # Even better (adaptive constraint)
            # effective_cost = tk_circ.n_gates * (2 ** n_qubits)
            # if effective_cost > threshold:
            #     reject

            # compiled = backend.get_compiled_circuit(tk_circ)

        except Exception as e:
            invalid_idxs.append(idx)
            errs.append(f"will_train() | {e}")
            continue

        valid.append(circ)

    
    print(f"In will_train() lost {len(invalid_idxs)} out of {len(circuits)}.")

    return valid, invalid_idxs, errs, n_qubits_list

def preprocess_and_encode(dataset: List[Dict]) -> Tuple[List[Dict], List[str], List[int]]:
    print(f"Preprocessing and encoding {len(dataset)} articles...")
    encoded_data, errors = [], []
    n_sentences = 0
    n_circuits = 0

    for i, data_dict in enumerate(tqdm(dataset, desc="Filtering and Encoding dataset")):
        with open(os.devnull, 'w') as devnull, \
         contextlib.redirect_stdout(devnull), \
         contextlib.redirect_stderr(devnull):

    # for i, data_dict in enumerate(dataset):
            text_sentences = data_dict["text_sentences"].copy()
            labels         = data_dict["labels"].deepcopy()

            n_text_sentences = len(text_sentences)
            n_labels = len(labels)

            sentences_simplified, remove = sentence_simplify_spacy(text_sentences)
            text_sentences               = remove_by_idx(text_sentences, remove)
            labels                       = remove_by_idx(labels, remove)

            diagrams, remove = sent2diagrams(sentences_simplified)
            diagrams         = remove_by_idx(diagrams, remove)
            text_sentences   = remove_by_idx(text_sentences, remove)
            labels           = remove_by_idx(labels, remove)


            normalized_diagrams, remove, errs2 = normalize(diagrams)
            text_sentences                     = remove_by_idx(text_sentences, remove)
            labels                             = remove_by_idx(labels, remove)


            circuits, remove, errs3 = quantum_encode(normalized_diagrams)
            text_sentences          = remove_by_idx(text_sentences, remove)
            labels                  = remove_by_idx(labels, remove)

            circuits, remove, errs4, n_qubits_list = will_train(circuits)
            text_sentences                         = remove_by_idx(text_sentences, remove)
            labels                                 = remove_by_idx(labels, remove)

            print("init:", n_text_sentences, n_labels,"\nafter:", len(circuits), len(text_sentences), len(labels))
            # n_qubits_list = [0]

            encoded_data.append(
                {
                    "circuits": circuits,
                    "labels": labels,
                    "original_text_sentences": text_sentences,
                }
            )

            errors += errs2 + errs3 + errs4 # + errs1 sent2diagrams has suppress_exceptions=True, so instead of errors, it returns None
            n_circuits += len(circuits)

    # errors.append(f"n_sentences: {n_text_sentences} | n_circuits: {n_circuits} | {n_text_sentences - n_circuits}")

    return encoded_data, errors, n_qubits_list



In [31]:
ld_ds = load_PreSumm_pts(file_number=0, ds_purpose="train", calculate_articles=True)

load_PreSumm_pts_0
No+: 0 | 1+ (>12): 135 | 2+ (>30): 0
Label distribution: 1: 147 | 2: 536 | 3: 1296 | 4: 0 | >4: 0
Positive labels ratio: 0.1818
Invalid ratio: 22 out of 2001 articles.

 n_sentences: 71022 
 n_labels: 71022 (average 35.8878) 
 n_labels_true: 5107 (average 2.5806)


In [14]:
def encode_save_auto(dataset: List[Dict], n_articles: int, left: int=0):
    right = 0
    while right < n_articles:
        right = min(left + 100, n_articles)
        print(left, "-", right)
        encoded_data, errors, n_qubits_list = preprocess_and_encode(dataset[left:right])

        encoded = {
            "encoded_dataset": encoded_data,
            "errors": errors,
            "n_qubits": n_qubits_list
        }
        with open(f"Dataset/Encoded/cnn_dailymail/PreSumm_{left}_{right}.pkl", 'wb') as file:
            pickle.dump(encoded, file)

        left = right

In [ ]:
datasetas = ld_ds.copy()
encode_save_auto(datasetas, n_articles, left=100)

In [ ]:
# 14m 27s for the first 101 | without compiled circuit int will_train() | 28 | 200 | 20000

# 101 - 201
# Preprocessing and encoding 100 articles...
# Filtering and Encoding dataset: 100%|██████████| 100/100 [33:47<00:00, 20.28s/it]
# 201 - 301
# Preprocessing and encoding 100 articles...
# Filtering and Encoding dataset: 100%|██████████| 100/100 [31:48<00:00, 19.09s/it]
# 301 - 401
# Preprocessing and encoding 100 articles...
# Filtering and Encoding dataset: 100%|██████████| 100/100 [33:41<00:00, 20.21s/it]
# 401 - 501
# Preprocessing and encoding 100 articles...
# Filtering and Encoding dataset: 100%|██████████| 100/100 [31:08<00:00, 18.69s/it]
# 501 - 601
# Preprocessing and encoding 100 articles...
# Filtering and Encoding dataset: 100%|██████████| 100/100 [35:07<00:00, 21.08s/it]
# 601 - 701
# Preprocessing and encoding 100 articles...
# Filtering and Encoding dataset: 100%|██████████| 100/100 [32:48<00:00, 19.68s/it]

In [15]:
def load_encoded_PreSumm_n_m(starting_index = 0, ending_index = 701):
    datasets_list = []
    for i in range(starting_index, ending_index, 100):
        with open(f"Dataset/Encoded/cnn_dailymail/PreSumm_{i}_{i + 100}.pkl", 'rb') as file:
            PreSum_loaded_ds = pickle.load(file)
            datasets_list.append(PreSum_loaded_ds)
    return datasets_list

# def combine_encoded_datasets(dataset1, dataset2):

def get_combined_encoded_dataset(dataset):
    return dataset["encoded_dataset"] if "encoded_dataset" in dataset else None

def get_combined_errors(dataset):
    return dataset["errors"] if "errors" in dataset else None

def load_encoded(filename = "Dataset/Encoded/cnn_dailymail/PreSumm_0_701.pkl"):
    with open(filename, 'rb') as file:
        loaded_encoded_ds = pickle.load(file)

    return loaded_encoded_ds

In [25]:
loaded_encoded_ds = load_encoded()

In [27]:
encoded_dataset = get_combined_encoded_dataset(loaded_encoded_ds)


In [26]:
# print(get_deep_type(encoded_dataset))

encoded_valid_labels = remove_negative_articles(loaded_encoded_ds["encoded_dataset"], "circuits", "labels", [0,1])

No+: 83 | 1+ (>12): 159 | 2+ (>30): 0
Label distribution: 1: 225 | 2: 266 | 3: 126 | 4: 0 | >4: 0
Positive labels ratio: 0.0400
Invalid ratio: 0 out of 700 articles.


In [20]:
print(get_deep_shape(loaded_encoded_ds))

dict(len=3)
  key='encoded_dataset' -> list(len=700)
    [0] -> dict(len=3)
      key='circuits' -> list(len=24)
        [*] -> Diagram
      key='labels' -> list(len=24)
        [*] -> list(len=2)
          [*] -> int
      key='original_text_sentences' -> list(len=24)
        [*] -> str
    [1] -> dict(len=3)
      key='circuits' -> list(len=31)
        [*] -> Diagram
      key='labels' -> list(len=31)
        [*] -> list(len=2)
          [*] -> int
      key='original_text_sentences' -> list(len=31)
        [*] -> str
    [2] -> dict(len=3)
      key='circuits' -> list(len=46)
        [*] -> Diagram
      key='labels' -> list(len=46)
        [*] -> list(len=2)
          [*] -> int
      key='original_text_sentences' -> list(len=46)
        [*] -> str
    [3] -> dict(len=3)
      key='circuits' -> list(len=7)
        [*] -> Diagram
      key='labels' -> list(len=7)
        [*] -> list(len=2)
          [*] -> int
      key='original_text_sentences' -> list(len=7)
        [*] -> str
  

In [53]:
# PreSum_0_100.keys()
for i, file_errors in enumerate(loaded_combined_errors):
    for error_message in file_errors:
        if "will_train()" not in error_message and "quantum_encode()" not in error_message and "normalize()" not in error_message:
            print(f"{i}: {error_message}")

0: n_sentences: 3601 | n_circuits: 2344 | 1257
1: n_sentences: 3416 | n_circuits: 2198 | 1218
2: n_sentences: 3473 | n_circuits: 2263 | 1210
3: n_sentences: 3371 | n_circuits: 2157 | 1214
4: n_sentences: 3764 | n_circuits: 2485 | 1279
5: n_sentences: 3503 | n_circuits: 2308 | 1195


In [16]:
import gc

# Delete large temporary variables
# del expensive_tensors

# Force Python to find unreferenced objects
gc.collect()

# Force the GPU to release the cached memory pool
torch.cuda.empty_cache()

In [ ]:
s = ["The president, speaking in Paris, announced sanctions while addressing the media.", 
     "My friend, a well-known scientist, published a paper.", 
     "The book that I read yesterday was fascinating.", 
     "He left the room while talking on the phone.", 
     "The CEO, who was under pressure, resigned after speaking to the board.", 
     "The president of the company announced reforms.", 
     "The cat sat on the mat."] 

expected_result = ["The president announced sanctions", "My friend published a paper", "The book was fascinating", "He left the room", "The CEO resigned", "The president of the company announced reforms", "The cat sat on the mat"]

for i, result in enumerate(sentence_simplify_spacy(s)[0]):
    print(s[i])
    print(result)
    print(expected_result[i], "\n")

After pruning: lost 0 / 7 sentences.
The president, speaking in Paris, announced sanctions while addressing the media.
The president announced sanctions
The president announced sanctions 

My friend, a well-known scientist, published a paper.
My friend published a paper
My friend published a paper 

The book that I read yesterday was fascinating.
The book was fascinating
The book was fascinating 

He left the room while talking on the phone.
He left the room
He left the room 

The CEO, who was under pressure, resigned after speaking to the board.
The CEO resigned after speaking to the board
The CEO resigned 

The president of the company announced reforms.
The president of the company announced reforms
The president of the company announced reforms 

The cat sat on the mat.
The cat sat on the mat
The cat sat on the mat 



In [ ]:
# s = combine_n_articles(ld_ds[:1], 1)[0]
s = ld_ds[:1][0]['text_sentences']
for i, result in enumerate(sentence_simplify_spacy(s)[0]):
    print(result)
    print(s[i], "\n")

After pruning: lost 0 / 36 sentences.
he 's on the outside and the view of viktor yanukovych seems to keep getting dimmer
he 's on the outside looking in , and the view of viktor yanukovych seems to keep getting dimmer . 

a news conference friday from his new quarters in southeastern russia underscored just how dim his prospects appear to be
a news conference friday from his new quarters in southeastern russia underscored just how dim his prospects appear to be , as the ousted ukrainian president complained that his host -- and potential benefactor -- was not around and had done little to make his stay more comfortable . 

i consider that russia must and has to act yanukovych told a phalanx of reporters
" i consider that russia must and has to act , " yanukovych told a phalanx of reporters who had assembled in the city of rostov-on-don , near the southwestern border with ukraine and about 700 miles south of moscow . 

he did not specify what actions he was hoping for but made clear wh

In [27]:
ld_ds_combined = combine_n_articles(ld_ds[:30], 2)
ld_ds_encoded, ld_ds_errors, ld_ds_n_qubits = preprocess_and_encode(ld_ds_combined)


Filtering and Encoding dataset: 100%|██████████| 15/15 [12:29<00:00, 49.94s/it]


In [28]:
encoded = {
    "encoded_dataset": ld_ds_encoded,
    "errors": ld_ds_errors,
    "n_qubits": ld_ds_n_qubits
}
with open('Dataset/Encoded/cnn_dailymail/mem_test_ds_vol4.pkl', 'wb') as file:
    pickle.dump(encoded, file)

In [20]:
ld_ds_combined = combine_n_articles(ld_ds[:30], 3)
ld_ds_encoded, ld_ds_errors, ld_ds_n_qubits = preprocess_and_encode(ld_ds_combined)

# encoded = {
#     "encoded_dataset": ld_ds_encoded,
#     "errors": ld_ds_errors,
#     "n_qubits": ld_ds_n_qubits
# }
# with open('Dataset/Encoded/cnn_dailymail/PreSumm_0_30_2_vol_2.pkl', 'wb') as file:
#     pickle.dump(encoded, file)

Filtering and Encoding dataset: 100%|██████████| 10/10 [11:41<00:00, 70.13s/it]


In [18]:
ld_ds_combined = combine_n_articles(ld_ds[:30], 10)
ld_ds_encoded, ld_ds_errors, ld_ds_n_qubits = preprocess_and_encode(ld_ds_combined)

# encoded = {
#     "encoded_dataset": ld_ds_encoded,
#     "errors": ld_ds_errors,
#     "n_qubits": ld_ds_n_qubits
# }
# with open('Dataset/Encoded/cnn_dailymail/PreSumm_0_30_30_vol_2.pkl', 'wb') as file:
#     pickle.dump(encoded, file)

Filtering and Encoding dataset: 100%|██████████| 3/3 [11:28<00:00, 229.60s/it]


In [21]:
print_length(ld_ds_combined)
print_length(ld_ds_encoded)

print_errors(ld_ds_errors)

n_text_sentences: 879 | n_lables: 879
n_circuits: 879 | n_lables: 879
error_list_length: 411
0: quantum_encode() | Ty(p)
1: quantum_encode() | Ty(p)
2: quantum_encode() | Ty(p)
3: quantum_encode() | Ty(p)
4: quantum_encode() | Ty(p)
5: quantum_encode() | Ty(p)
6: quantum_encode() | Ty(p)
7: quantum_encode() | Ty(p)
8: quantum_encode() | Ty(p)
9: quantum_encode() | Ty(p)
10: quantum_encode() | Ty(p)
11: quantum_encode() | Ty(p)
12: quantum_encode() | Ty(p)
13: quantum_encode() | Ty(p)
14: quantum_encode() | Ty(p)
15: quantum_encode() | Ty(p)
16: quantum_encode() | Ty(p)
17: quantum_encode() | Ty(p)
18: quantum_encode() | Ty(p)
19: quantum_encode() | Ty(p)
20: quantum_encode() | Ty(p)
21: quantum_encode() | Ty(p)
22: quantum_encode() | Ty(p)
23: quantum_encode() | Ty(p)
24: quantum_encode() | Ty(p)
25: quantum_encode() | Ty(p)
26: quantum_encode() | Ty(p)
27: quantum_encode() | Ty(p)
28: quantum_encode() | Ty(p)
29: quantum_encode() | Ty(p)
30: quantum_encode() | Ty(p)
31: quantum_encode

In [ ]:
ld_ds_combined3 = combine_n_articles(ld_ds[:30], 10)
ld_ds_encoded3, ld_ds_errors3, ld_ds_n_qubits3 = preprocess_and_encode(ld_ds_combined3)

In [ ]:
encoded3 = {
    "encoded_dataset": ld_ds_encoded3,
    "errors": ld_ds_errors3,
    "n_qubits": ld_ds_n_qubits3
}
with open('Dataset/Encoded/cnn_dailymail/PreSumm_0_30_10.pkl', 'wb') as file:
    pickle.dump(encoded3, file)

In [ ]:
print_length(ld_ds_combined3)
print_length(ld_ds_encoded3)

print_errors(ld_ds_errors3)

In [ ]:
encoded_dataset, errors, n_qubits = preprocess_and_encode(ld_ds_combined)
# On GPU it took ~4m 36.0s to encode first 10 PreSumm articles seperately without will_train()
# On GPU it took ~14m 39.4s to encode first 30 PreSumm articles seperately without will_train()
# On GPU it took ~14m 27.1s to encode first 30 PreSumm articles (merged 2:1) without will_train()
# On GPU it took ~14m 27.6s to encode first 30 PreSumm articles (merged 3:1) without will_train()
# On GPU it took ~14m 54.1s to encode first 30 PreSumm articles (merged 6:1) without will_train()
# On GPU it took ~15m 40.0s to encode first 30 PreSumm articles (merged 10:1) without will_train()

# On GPU it took ~24m 30.0s to encode first 30 PreSumm articles seperately
# On GPU it took ~24m 32.6s to encode first 30 PreSumm articles (merged 2:1)
# On GPU it took ~24m 24.7s to encode first 30 PreSumm articles (merged 3:1)
# On GPU it took ~24m 46.4s to encode first 30 PreSumm articles (merged 6:1)
# On GPU it took ~25m 50.s to encode first 30 PreSumm articles (merged 10:1)
# On GPU it took ~27m 54.s to encode first 30 PreSumm articles (merged 30:1)

# On GPU it took ~24m 14s to encode first 30 PreSumm articles seperately with optimized will_train and AerSimulator object
# On GPU it took ~24m 30s to encode first 30 PreSumm articles (merged 2:1) with optimized will_train and AerSimulator object
# On GPU it took ~23m 56s to encode first 30 PreSumm articles (merged 3:1) with optimized will_train and AerSimulator object 2
# On GPU it took ~15m 46s to encode first 30 PreSumm articles (merged 3:1) with optimized will_train and AerSimulator object 1
# On GPU it took ~24m 54s to encode first 30 PreSumm articles (merged 6:1)  with optimized will_train and AerSimulator object 2
# On GPU it took ~17m 1s to encode first 30 PreSumm articles (merged 6:1)  with optimized will_train and AerSimulator object 1
# On GPU it took ~26m 0s to encode first 30 PreSumm articles (merged 10:1)  with optimized will_train and AerSimulator object 2
# On GPU it took ~m .s to encode first 30 PreSumm articles (merged 30:1)  with optimized will_train and AerSimulator object

# Encode on GPU | ~12m 15s | 0-30 PreSumm | 1:1  | SpaCy pruning | Rewriter adjustments | Optimized will_train, AerSimulator 
# Encode on GPU | ~11m 29s | 0-30 PreSumm | 1:2  | SpaCy pruning | Rewriter adjustments | Optimized will_train, AerSimulator 
# Encode on GPU | ~11m 38s | 0-30 PreSumm | 1:3  | SpaCy pruning | Rewriter adjustments | Optimized will_train, AerSimulator 
# Encode on GPU | ~11m 28s | 0-30 PreSumm | 1:10 | SpaCy pruning | Rewriter adjustments | Optimized will_train, AerSimulator 
# Encode on GPU | ~11m 25s | 0-30 PreSumm | 1:30 | SpaCy pruning | Rewriter adjustments | Optimized will_train, AerSimulator 


In [ ]:
print_length(encoded_dataset)
print_n_qubits(n_qubits)
print_errors(errors)

In [ ]:
encoded = {
    "encoded_dataset": encoded_dataset,
    "errors": errors,
    "n_qubits": n_qubits
}
with open('Dataset/Encoded/cnn_dailymail/PreSumm_0_30_1.pkl', 'wb') as file:
    pickle.dump(encoded, file)

In [ ]:
with open('Dataset/Encoded/cnn_dailymail/PreSumm_0_30_1.pkl', "rb") as file:
    encoded_loaded = pickle.load(file)

In [ ]:
def s2d(raw_ds):
    diagrams = []
    for data_dict in tqdm(raw_ds):
        with open(os.devnull, 'w') as devnull, \
         contextlib.redirect_stdout(devnull), \
         contextlib.redirect_stderr(devnull):
            # diagrams.append(sent2diagrams(data_dict["text_sentences"]))
            sent2diagrams(data_dict["text_sentences"])

    return diagrams

x_ds = combine_n_articles(ld_ds[:200], 1)
d = s2d(x_ds)

# 32m 50s single articles over 200 in sent2diagrams
# 37m 28s 5:1 article over 200 in sent2diagrams
# 43m 24s 20:1 articles over 200 in sent2diagrams
# 84m 19s single articles over 200 in sent2diagrams_single

In [ ]:
# 87
# 0: Diagram 0 (cod=s @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ p.l @ p @ n.l @ n @ n.l @ n @ n.l @ n @ s.r @ s @ s.l @ conj.l @ conj @ conj.l) does not compose with diagram 1 (dom=s @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ p.l @ p @ n.l @ n @ n.l @ n @ n.l @ n @ s.r @ s @ s.l @ conj.l @ conj @ conj.l @ conj) ( normalize() )
# 1: Diagram 0 (cod=n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n.l.l @ n.l @ n @ n.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ s.r @ n.r.r @ n.r @ s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ s.r @ n.r.r @ n.r @ s @ s.l @ n @ conj.l @ conj @ conj.l) does not compose with diagram 1 (dom=n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n.l.l @ n.l @ n @ n.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ s.r @ n.r.r @ n.r @ s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ s.r @ n.r.r @ n.r @ s @ s.l @ n @ conj.l @ conj @ conj.l @ conj) ( normalize() )
# 2: Diagram 0 (cod=s @ s.l @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n @ n.r @ s @ s.r @ n.r.r @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l) does not compose with diagram 1 (dom=s @ s.l @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n @ n.r @ s @ s.r @ n.r.r @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l @ conj) ( normalize() )
# 3: Diagram 0 (cod=s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l) does not compose with diagram 1 (dom=s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l @ conj) ( normalize() )
# 4: 37 qubits exceeds safe limit 28. ( will_train() )
# 86: 40 qubits exceeds safe limit 28. ( will_train() )
# [37, 31, 53, 13, 23, 14, 14, 14, 13, 17, 13, 7, 18, 13, 20, 17, 5, 21, 13, 9, 13, 11, 9, 17, 10, 27, 15, 14, 7, 18, 22, 16, 20, 13, 9, 34, 12, 13, 23, 26, 45, 14, 21, 7, 19, 7, 52, 12, 9, 17, 18, 13, 12, 26, 23, 16, 12, 11, 12, 11, 27, 26, 32, 38, 31, 26, 43, 29, 5, 5, 18, 24, 36, 20, 20, 35, 24, 40, 18, 22, 23, 42, 25, 12, 23, 28, 15, 17, 38, 49, 36, 22, 13, 27, 29, 18, 29, 16, 15, 31, 26, 20, 6, 17, 19, 32, 5, 36, 22, 18, 17, 8, 18, 16, 26, 11, 13, 14, 15, 7, 8, 5, 11, 18, 24, 17, 14, 5, 12, 11, 7, 29, 11, 34, 20, 38, 24, 13, 26, 25, 22, 38, 49, 24, 46, 21, 36, 16, 8, 13, 35, 17, 41, 33, 33, 31, 35, 49, 22, 24, 51, 37, 41, 36, 37, 32, 28, 22, 27, 28, 16, 43, 20, 17, 30, 27, 47, 49, 22, 36, 31, 40, 25, 17, 27, 47, 53, 46, 25, 15, 13, 18, 24, 22, 34, 31, 21, 18, 20, 43, 59, 13, 29, 12, 14, 20, 15, 47, 46, 31, 15, 38, 50, 17, 39, 23, 28, 42, 16, 49, 42, 32, 49, 21, 26, 51, 18, 11, 41, 23, 33, 30, 5, 11, 16, 14, 21, 27, 22, 13, 27, 23, 30, 27, 28, 35, 34, 23, 16, 11, 22, 29, 5, 10, 18, 24, 20, 21, 42, 28, 50, 36, 42, 14, 13, 27, 42, 40, 16]
# length: 269
# 23.966542750929367

In [ ]:
# ld_ds[:30], 3
# 60
# 0: Diagram 0 (cod=n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n.l.l @ n.l @ n @ n.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ s.r @ n.r.r @ n.r @ s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ s.r @ n.r.r @ n.r @ s @ s.l @ n @ conj.l @ conj @ conj.l) does not compose with diagram 1 (dom=n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n.l.l @ n.l @ n @ n.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ s.r @ n.r.r @ n.r @ s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ s.r @ n.r.r @ n.r @ s @ s.l @ n @ conj.l @ conj @ conj.l @ conj) ( normalize() )
# 1: Diagram 0 (cod=s @ s.l @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n @ n.r @ s @ s.r @ n.r.r @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l) does not compose with diagram 1 (dom=s @ s.l @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n @ n.r @ s @ s.r @ n.r.r @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l @ conj) ( normalize() )
# 2: Diagram 0 (cod=s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l) does not compose with diagram 1 (dom=s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l @ conj) ( normalize() )
# 3: 38 qubits exceeds safe limit 28. ( will_train() )
# 59: 40 qubits exceeds safe limit 28. ( will_train() )
# [22, 38, 49, 24, 46, 21, 36, 16, 8, 13, 35, 17, 41, 33, 33, 31, 35, 49, 22, 24, 51, 37, 41, 36, 37, 32, 28, 22, 27, 28, 16, 43, 20, 17, 30, 27, 47, 49, 22, 36, 31, 40, 25, 17, 27, 47, 53, 46, 25, 15, 13, 18, 24, 22, 34, 31, 21, 18, 20, 43, 59, 13, 29, 12, 14, 20, 15, 47, 46, 31, 15, 38, 50, 17, 39, 23, 28, 42, 16, 49, 42, 32, 49, 21, 26, 51, 18, 11, 41, 23, 33, 30, 5, 11, 16, 14, 21, 27, 22, 13, 27, 23, 30, 27, 28, 35, 34, 23, 16, 11, 22, 29, 5, 10, 18, 24, 20, 21, 42, 28, 50, 36, 42, 14, 13, 27, 42, 40, 16]
# length: 129
# 28.45736434108527

In [ ]:
# 149
# 0: Diagram 0 (cod=n @ n.l @ n @ n.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ p.l @ n.l @ n) does not compose with diagram 1 (dom=n @ n.l @ n @ n.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ p.l @ n.l @ n @ n) ( normalize() )
# 1: Diagram 0 (cod=n @ n.l @ n @ n.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ s.r @ n.r.r @ n.r @ s @ p.l) does not compose with diagram 1 (dom=n @ n.l @ n @ n.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ s.r @ n.r.r @ n.r @ s @ p.l @ n) ( normalize() )
# 2: Diagram 0 (cod=s @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ p.l @ p @ n.l @ n @ n.l @ n @ n.l @ n @ s.r @ s @ s.l @ conj.l @ conj @ conj.l) does not compose with diagram 1 (dom=s @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ p.l @ p @ n.l @ n @ n.l @ n @ n.l @ n @ s.r @ s @ s.l @ conj.l @ conj @ conj.l @ conj) ( normalize() )
# 3: Diagram 0 (cod=n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n.l.l @ n.l @ n @ n.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ s.r @ n.r.r @ n.r @ s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ s.r @ n.r.r @ n.r @ s @ s.l @ n @ conj.l @ conj @ conj.l) does not compose with diagram 1 (dom=n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n.l.l @ n.l @ n @ n.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ s.r @ n.r.r @ n.r @ s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ s.r @ n.r.r @ n.r @ s @ s.l @ n @ conj.l @ conj @ conj.l @ conj) ( normalize() )
# 4: Diagram 0 (cod=s @ s.l @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n @ n.r @ s @ s.r @ n.r.r @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l) does not compose with diagram 1 (dom=s @ s.l @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n @ n.r @ s @ s.r @ n.r.r @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l @ conj) ( normalize() )
# 5: Diagram 0 (cod=s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l) does not compose with diagram 1 (dom=s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l @ conj) ( normalize() )
# 6: 29 qubits exceeds safe limit 28. ( will_train() )
# 148: 40 qubits exceeds safe limit 28. ( will_train() )
# [5, 5, 26, 29, 30, 19, 58, 30, 35, 34, 55, 20, 71, 26, 30, 59, 26, 16, 25, 17, 40, 39, 30, 28, 32, 25, 5, 5, 18, 30, 32, 5, 11, 11, 45, 39, 26, 29, 26, 13, 23, 7, 18, 32, 14, 11, 26, 23, 15, 6, 4, 11, 14, 18, 7, 14, 20, 14, 35, 35, 14, 26, 25, 16, 13, 25, 12, 14, 18, 24, 35, 36, 19, 15, 10, 47, 31, 17, 16, 33, 24, 18, 30, 32, 37, 41, 36, 33, 18, 52, 45, 57, 46, 28, 38, 14, 20, 13, 47, 27, 42, 38, 31, 36, 24, 53, 15, 34, 40, 12, 31, 30, 5, 5, 43, 45, 48, 27, 10, 18, 50, 38, 30, 76, 19, 19, 20, 22, 29, 28, 23, 23, 30, 19, 28, 16, 30, 37, 25, 39, 37, 31, 53, 13, 23, 14, 14, 14, 13, 17, 13, 7, 18, 13, 20, 17, 5, 21, 13, 9, 13, 11, 9, 17, 10, 27, 15, 14, 7, 18, 22, 16, 20, 13, 9, 34, 12, 13, 23, 26, 45, 14, 21, 7, 19, 7, 52, 12, 9, 17, 18, 13, 12, 26, 23, 16, 12, 11, 12, 11, 27, 26, 32, 38, 31, 26, 43, 29, 5, 5, 18, 24, 36, 20, 20, 35, 24, 40, 18, 22, 23, 42, 25, 12, 23, 28, 15, 17, 38, 49, 36, 22, 13, 27, 29, 18, 29, 16, 15, 31, 26, 20, 6, 17, 19, 32, 5, 36, 22, 18, 17, 8, 18, 16, 26, 11, 13, 14, 15, 7, 8, 5, 11, 18, 24, 17, 14, 5, 12, 11, 7, 29, 11, 34, 20, 38, 24, 13, 26, 25, 22, 38, 49, 24, 46, 21, 36, 16, 8, 13, 35, 17, 41, 33, 33, 31, 35, 49, 22, 24, 51, 37, 41, 36, 37, 32, 28, 22, 27, 28, 16, 43, 20, 17, 30, 27, 47, 49, 22, 36, 31, 40, 25, 17, 27, 47, 53, 46, 25, 15, 13, 18, 24, 22, 34, 31, 21, 18, 20, 43, 59, 13, 29, 12, 14, 20, 15, 47, 46, 31, 15, 38, 50, 17, 39, 23, 28, 42, 16, 49, 42, 32, 49, 21, 26, 51, 18, 11, 41, 23, 33, 30, 5, 11, 16, 14, 21, 27, 22, 13, 27, 23, 30, 27, 28, 35, 34, 23, 16, 11, 22, 29, 5, 10, 18, 24, 20, 21, 42, 28, 50, 36, 42, 14, 13, 27, 42, 40, 16]
# length: 409
# 24.9119804400978

In [ ]:
# 14
# 0: ( will_train() ) 51 qubits exceeds safe limit 28.
# 13: ( will_train() ) 40 qubits exceeds safe limit 28.
# [21, 26, 51, 18, 11, 41, 23, 33, 30, 5, 11, 16, 14, 21, 27, 22, 13, 27, 23, 30, 27, 28, 35, 34, 23, 16, 11, 22, 29, 5, 10, 18, 24, 20, 21, 42, 28, 50, 36, 42, 14, 13, 27, 42, 40, 16]
# length: 46
# 24.695652173913043

In [ ]:
# 38
# 0: ( normalize() ) Diagram 0 (cod=s @ s.l @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n @ n.r @ s @ s.r @ n.r.r @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l) does not compose with diagram 1 (dom=s @ s.l @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n @ n.r @ s @ s.r @ n.r.r @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l @ conj)
# 1: ( normalize() ) Diagram 0 (cod=s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l) does not compose with diagram 1 (dom=s @ s.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ conj.l @ conj @ conj.l @ conj)
# 2: ( will_train() ) 36 qubits exceeds safe limit 28.
# 37: ( will_train() ) 40 qubits exceeds safe limit 28.
# [36, 31, 40, 25, 17, 27, 47, 53, 46, 25, 15, 13, 18, 24, 22, 34, 31, 21, 18, 20, 43, 59, 13, 29, 12, 14, 20, 15, 47, 46, 31, 15, 38, 50, 17, 39, 23, 28, 42, 16, 49, 42, 32, 49, 21, 26, 51, 18, 11, 41, 23, 33, 30, 5, 11, 16, 14, 21, 27, 22, 13, 27, 23, 30, 27, 28, 35, 34, 23, 16, 11, 22, 29, 5, 10, 18, 24, 20, 21, 42, 28, 50, 36, 42, 14, 13, 27, 42, 40, 16]
# length: 90
# 27.42222222222222

In [ ]:
# on CPU it takes ~5m 56.5s to encode first 10 articles
# on `GPU` it takes ~4m 57.5s to encode first 10 articles

# on CPU and feeding whole articles (not single sentences) it takes ~3m 4.2s to encode first 10 articles
# on `GPU` and feeding whole articles (not single sentences) it takes ~2m 44.2s to encode first 10 articles

# on CPU and feeding multiple (40) articles it takes ~_m _._s
# on `GPU` and feeding multiple (40) articles it takes ~15m 21.3s

# on CPU and feeding whole articles (not single sentences) it takes ~m s to encode first 40 articles
# on `GPU` and feeding whole articles (not single sentences) it takes ~14m 49s to encode first 40 articles
